# Newsroom — Preprocessing

This notebook handles data loading, cleaning, and preparation for the Newsroom analysis.


In [17]:
import pandas as pd
import numpy as np
import ast
import re


## Load Raw Data


In [18]:
# Load raw data
df = pd.read_csv('../data/original_data/newsroom_judged.csv')

print("=" * 50)
print("RAW DATA")
print("=" * 50)
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()


RAW DATA
Shape: (420, 29)
Columns: ['Unnamed: 0', 'id', 'instance', 'informativeness_mean', 'informativeness_scores', 'relevance_mean', 'relevance_scores', 'fluency_mean', 'fluency_scores', 'coherence_mean', 'coherence_scores', 'SystemSummary', 'ArticleText', 'GPT_informativeness_as_a_judge', 'GPT_relevance_as_a_judge', 'GPT_fluency_as_a_judge', 'GPT_coherence_as_a_judge', 'LLAMA_informativeness_as_a_judge', 'LLAMA_relevance_as_a_judge', 'LLAMA_fluency_as_a_judge', 'LLAMA_coherence_as_a_judge', 'MISTRAL_informativeness_as_a_judge', 'MISTRAL_relevance_as_a_judge', 'MISTRAL_fluency_as_a_judge', 'MISTRAL_coherence_as_a_judge', 'MIXTRAL_informativeness_as_a_judge', 'MIXTRAL_relevance_as_a_judge', 'MIXTRAL_fluency_as_a_judge', 'MIXTRAL_coherence_as_a_judge']


,Unnamed: 0,id,instance,informativeness_mean,informativeness_scores,relevance_mean,relevance_scores,fluency_mean,fluency_scores,coherence_mean,...,LLAMA_fluency_as_a_judge,LLAMA_coherence_as_a_judge,MISTRAL_informativeness_as_a_judge,MISTRAL_relevance_as_a_judge,MISTRAL_fluency_as_a_judge,MISTRAL_coherence_as_a_judge,MIXTRAL_informativeness_as_a_judge,MIXTRAL_relevance_as_a_judge,MIXTRAL_fluency_as_a_judge,MIXTRAL_coherence_as_a_judge
0,0,1,### Generated Summary collection of all usato...,2.67,"[4, 3, 1]",3.33,"[4, 5, 1]",3.67,"[3, 5, 3]",3.67,...,1,1,1 (low),1 (low),1 (low),1,"I would give the summary a 1 out of 5, as it d...",I would give the summary a 1 out of 5 for its ...,I would give the summary you provided a rating...,I would give the summary a 1 out of 5 for not ...
1,1,2,"### Generated Summary Jacksonville , Ark. , p...",4.33,"[4, 5, 4]",4.67,"[4, 5, 5]",4.33,"[3, 5, 5]",4.00,...,4,4,4.5,5,5,5,"Based on the information provided, I would giv...",5 (high) \n\nThe details provided in the summa...,I would give the individual sentences of the s...,"Based on the given text, I would give the summ..."
2,2,3,"### Generated Summary Jacksonville , Ark. , p...",4.00,"[3, 5, 4]",4.00,"[4, 5, 3]",4.00,"[4, 5, 3]",4.33,...,4,4,3.5 (The summary captures the main points of t...,4.5 (The summary is very consistent with the d...,5 (high),4.5 (The summary is concise and accurately rep...,I would give the summary a 3 out of 5 for capt...,5\n\nThe summary provides a clear and accurate...,I would give the individual sentences of this ...,I would give the summary a 4 out of 5 for fitt...
3,3,4,### Generated Summary stars joshua rendon 16 ...,3.00,"[3, 3, 3]",3.67,"[3, 4, 4]",3.00,"[3, 2, 4]",2.67,...,2,2,2,3,2,2,I would give the summary a rating of 2 out of ...,I would give the summary a rating of 2 out of ...,I would give the individual sentences in the s...,I would give the summary a 2 out of 5. The sum...
4,4,5,### Generated Summary joshua rendon and ebony...,4.00,"[3, 4, 5]",3.33,"[4, 3, 3]",3.67,"[4, 3, 4]",3.67,...,1,1,2,3,1 (low) - The summary is incomplete and lacks ...,1,I would give the summary a 2 out of 5 for its ...,5\n\nThe summary accurately captures the main ...,I would give the individual sentences of this ...,I would give the summary a 3 out of 5 for fit ...


In [19]:
# Keep only relevant columns
columns_to_keep = [
    # Human Judgements
    'informativeness_scores', 'relevance_scores', 'fluency_scores', 'coherence_scores',
    # GPT
    'GPT_informativeness_as_a_judge', 'GPT_relevance_as_a_judge',
    'GPT_fluency_as_a_judge', 'GPT_coherence_as_a_judge',
    # LLAMA
    'LLAMA_informativeness_as_a_judge', 'LLAMA_relevance_as_a_judge',
    'LLAMA_fluency_as_a_judge', 'LLAMA_coherence_as_a_judge',
    # MISTRAL
    'MISTRAL_informativeness_as_a_judge', 'MISTRAL_relevance_as_a_judge',
    'MISTRAL_fluency_as_a_judge', 'MISTRAL_coherence_as_a_judge'
]



df = df.loc[:, columns_to_keep]

print(f"Shape after keeping relevant columns: {df.shape}")
print(f"Columns: {list(df.columns)}")


Shape after keeping relevant columns: (420, 16)
Columns: ['informativeness_scores', 'relevance_scores', 'fluency_scores', 'coherence_scores', 'GPT_informativeness_as_a_judge', 'GPT_relevance_as_a_judge', 'GPT_fluency_as_a_judge', 'GPT_coherence_as_a_judge', 'LLAMA_informativeness_as_a_judge', 'LLAMA_relevance_as_a_judge', 'LLAMA_fluency_as_a_judge', 'LLAMA_coherence_as_a_judge', 'MISTRAL_informativeness_as_a_judge', 'MISTRAL_relevance_as_a_judge', 'MISTRAL_fluency_as_a_judge', 'MISTRAL_coherence_as_a_judge']


In [20]:
# reanme judge columns to have consistent naming
rename_dict = {
    'GPT_informativeness_as_a_judge': 'GPT-4o_informativeness_as_a_judge',
    'GPT_relevance_as_a_judge': 'GPT-4o_relevance_as_a_judge',
    'GPT_fluency_as_a_judge': 'GPT-4o_fluency_as_a_judge',
    'GPT_coherence_as_a_judge': 'GPT-4o_coherence_as_a_judge',

    'LLAMA_informativeness_as_a_judge': 'Llama_informativeness_as_a_judge',
    'LLAMA_relevance_as_a_judge': 'Llama_relevance_as_a_judge',
    'LLAMA_fluency_as_a_judge': 'Llama_fluency_as_a_judge',
    'LLAMA_coherence_as_a_judge': 'Llama_coherence_as_a_judge', 

    'MISTRAL_informativeness_as_a_judge': 'Mistral_informativeness_as_a_judge',
    'MISTRAL_relevance_as_a_judge': 'Mistral_relevance_as_a_judge',
    'MISTRAL_fluency_as_a_judge': 'Mistral_fluency_as_a_judge',
    'MISTRAL_coherence_as_a_judge': 'Mistral_coherence_as_a_judge'
}

df = df.rename(columns=rename_dict)

## Process Human Scores

Convert human score strings to lists and verify all have 3 judges.


In [21]:
# Get human score columns
human_scores = [col for col in df.columns if 'scores' in col]

# Safely evaluate string representations of lists
for col in human_scores:
    df[col] = df[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

# Verify all lists have 3 judges
print("Checking human score list lengths:")
for col in human_scores:
    lengths = df[col].apply(len)
    print(f"\n{col}:")
    print(lengths.value_counts())
    all_same_length = lengths.nunique() == 1
    print(f"All lists same length? {all_same_length}")


Checking human score list lengths:

informativeness_scores:
informativeness_scores
3    420
Name: count, dtype: int64
All lists same length? True

relevance_scores:
relevance_scores
3    420
Name: count, dtype: int64
All lists same length? True

fluency_scores:
fluency_scores
3    420
Name: count, dtype: int64
All lists same length? True

coherence_scores:
coherence_scores
3    420
Name: count, dtype: int64
All lists same length? True


In [22]:
# Break up human judgements into separate columns (human_#1, human_#2, human_#3)
for col in human_scores:
    metric_name = col.split('_')[0]
    for i in range(1, 4):
        new_col_name = f'{metric_name}_human_#{i}'
        df[new_col_name] = df[col].apply(lambda x: x[i-1])

# Drop original list columns
df.drop(columns=human_scores, inplace=True)

print(f"Shape after expanding human scores: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head(3)


Shape after expanding human scores: (420, 24)
Columns: ['GPT-4o_informativeness_as_a_judge', 'GPT-4o_relevance_as_a_judge', 'GPT-4o_fluency_as_a_judge', 'GPT-4o_coherence_as_a_judge', 'Llama_informativeness_as_a_judge', 'Llama_relevance_as_a_judge', 'Llama_fluency_as_a_judge', 'Llama_coherence_as_a_judge', 'Mistral_informativeness_as_a_judge', 'Mistral_relevance_as_a_judge', 'Mistral_fluency_as_a_judge', 'Mistral_coherence_as_a_judge', 'informativeness_human_#1', 'informativeness_human_#2', 'informativeness_human_#3', 'relevance_human_#1', 'relevance_human_#2', 'relevance_human_#3', 'fluency_human_#1', 'fluency_human_#2', 'fluency_human_#3', 'coherence_human_#1', 'coherence_human_#2', 'coherence_human_#3']


,GPT-4o_informativeness_as_a_judge,GPT-4o_relevance_as_a_judge,GPT-4o_fluency_as_a_judge,GPT-4o_coherence_as_a_judge,Llama_informativeness_as_a_judge,Llama_relevance_as_a_judge,Llama_fluency_as_a_judge,Llama_coherence_as_a_judge,Mistral_informativeness_as_a_judge,Mistral_relevance_as_a_judge,...,informativeness_human_#3,relevance_human_#1,relevance_human_#2,relevance_human_#3,fluency_human_#1,fluency_human_#2,fluency_human_#3,coherence_human_#1,coherence_human_#2,coherence_human_#3
0,Evaluation Error,Evaluation Error,Evaluation Error,Evaluation Error,1,1,1,1,1 (low),1 (low),...,1,4,5,1,3,5,3,4,4,3
1,Evaluation Error,Evaluation Error,Evaluation Error,Evaluation Error,4,5,4,4,4.5,5,...,4,4,5,5,3,5,5,3,5,4
2,Evaluation Error,Evaluation Error,Evaluation Error,Evaluation Error,3,4,4,4,3.5 (The summary captures the main points of t...,4.5 (The summary is very consistent with the d...,...,4,4,5,3,4,5,3,4,5,4


## Clean LLM Scores

Extract numerical ratings from LLM text outputs.


In [23]:
def extract_valid_rating(series: pd.Series) -> pd.Series:
    """
    Extract leading number from strings in a Series,
    keeping only values between 1 and 5 inclusive.
    Non-matching or out-of-range values are returned as NaN.

    Parameters
    ----------
    series : pd.Series

    Returns
    -------
    pd.Series
        Series of integers in [1, 5] or NaN.
    """
    # Pattern matches optional whitespace + number with optional decimal part
    pattern = re.compile(r'^\s*(\d+(?:\.\d+)?)')

    values = []
    for item in series:
        text = str(item)
        match = pattern.match(text)
        if match:
            value = float(match.group(1))
            if 1 <= value <= 5:
                values.append(value)
            else:
                values.append(np.nan)
        else:
            values.append(np.nan)

    return pd.Series(values, index=series.index)


In [24]:
# Get model columns
model_columns = [col for col in df.columns if 'as_a_judge' in col]

# Drop rows where any model column contains 'Evaluation Error'
mask = df[model_columns].apply(lambda row: row.astype(str).str.contains("Evaluation Error")).any(axis=1)
rows_before = len(df)
df = df[~mask]
print(f"Dropped {rows_before - len(df)} rows with 'Evaluation Error'")

# Drop NaN values
rows_before = len(df)
df = df.dropna()
print(f"Dropped {rows_before - len(df)} rows with NaN values")

# Apply cleaning and convert to int
for col in model_columns:
    df[col] = extract_valid_rating(df[col])
    df[col] = df[col].round().astype("Int64")

print(f"\nFinal shape: {df.shape}")
df.head(3)


Dropped 93 rows with 'Evaluation Error'
Dropped 0 rows with NaN values

Final shape: (327, 24)


,GPT-4o_informativeness_as_a_judge,GPT-4o_relevance_as_a_judge,GPT-4o_fluency_as_a_judge,GPT-4o_coherence_as_a_judge,Llama_informativeness_as_a_judge,Llama_relevance_as_a_judge,Llama_fluency_as_a_judge,Llama_coherence_as_a_judge,Mistral_informativeness_as_a_judge,Mistral_relevance_as_a_judge,...,informativeness_human_#3,relevance_human_#1,relevance_human_#2,relevance_human_#3,fluency_human_#1,fluency_human_#2,fluency_human_#3,coherence_human_#1,coherence_human_#2,coherence_human_#3
7,1,1,1,1,2,2,1,2,4,1,...,1,4,1,1,5,3,1,5,1,1
8,1,1,1,1,2,2,2,3,4,3,...,4,4,2,3,4,3,3,4,1,4
9,1,1,2,1,2,2,2,2,5,5,...,5,2,5,4,3,5,5,2,2,5


In [25]:
# Check for remaining null values
print("Null values per column:")
print(df.isnull().sum())

Null values per column:
GPT-4o_informativeness_as_a_judge     0
GPT-4o_relevance_as_a_judge           0
GPT-4o_fluency_as_a_judge             0
GPT-4o_coherence_as_a_judge           0
Llama_informativeness_as_a_judge      0
Llama_relevance_as_a_judge            0
Llama_fluency_as_a_judge              0
Llama_coherence_as_a_judge            0
Mistral_informativeness_as_a_judge    4
Mistral_relevance_as_a_judge          4
Mistral_fluency_as_a_judge            0
Mistral_coherence_as_a_judge          0
informativeness_human_#1              0
informativeness_human_#2              0
informativeness_human_#3              0
relevance_human_#1                    0
relevance_human_#2                    0
relevance_human_#3                    0
fluency_human_#1                      0
fluency_human_#2                      0
fluency_human_#3                      0
coherence_human_#1                    0
coherence_human_#2                    0
coherence_human_#3                    0
dtype: int64


## Save Cleaned Data


In [26]:
# Save cleaned dataset
df.to_csv("../data/newsroom_judged_cleaned.csv", index=False)
print(f"✅ Saved cleaned dataset to ../data/newsroom_judged_cleaned.csv")
print(f"   Shape: {df.shape}")

print()
print("=" * 50)
print("PREPROCESSING COMPLETE")
print("=" * 50)
print()
print("Output file:")
print("  - ../data/newsroom_judged_cleaned.csv")


✅ Saved cleaned dataset to ../data/newsroom_judged_cleaned.csv
   Shape: (327, 24)

PREPROCESSING COMPLETE

Output file:
  - ../data/newsroom_judged_cleaned.csv
